# Investigation 01 — v1 vs v2 paired comparison

This notebook tests the effect of the v2 abstention prompt as a paired intervention.

Instead of comparing aggregate averages only, it aligns v1 and v2 predictions for the same:

`provider × model × prompt family × condition × sentence × repetition`

This makes it possible to answer:

- Which exact v1 errors were fixed by v2?
- Did v2 introduce any new errors?
- Did v2 mainly improve controls, or also improve schema-present examples?
- Was the gain stronger for direct-schema prompting or structured-role prompting?

Thesis use: this directly supports the claim that the v2 schema-present/NONE mechanism is an experimental intervention rather than a minor prompt wording change.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from the project notebooks/ directory.
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = DATA_DIR / "outputs"
PARSED_PATH = OUTPUTS_DIR / "parsed_responses.jsonl"
RAW_PATH = OUTPUTS_DIR / "raw_responses.jsonl"
GOLD_PATH = DATA_DIR / "gold" / "sentences_v1.jsonl"

EXPORT_DIR = OUTPUTS_DIR / "top4_investigations"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON in {path} line {line_no}: {exc}") from exc
    return pd.DataFrame(rows)

def safe_read_jsonl(path: Path) -> pd.DataFrame:
    return read_jsonl(path) if path.exists() else pd.DataFrame()

def prompt_generation(prompt_id) -> str:
    prompt_id = str(prompt_id)
    if "v2" in prompt_id or "abstention" in prompt_id:
        return "v2_abstention"
    if "v1" in prompt_id:
        return "v1"
    return "unknown"

def prompt_base(prompt_id: str) -> str:
    prompt_id = str(prompt_id)
    if "direct_schema" in prompt_id:
        return "direct_schema"
    if "structured_roles" in prompt_id:
        return "structured_roles"
    if "naive" in prompt_id:
        return "naive"
    return "unknown"

def condition_family_from_id(condition_id) -> str:
    condition_id = str(condition_id)
    if "temp_0" in condition_id:
        return "temp_0"
    if "temp_03" in condition_id:
        return "temp_03"
    if "temp_07" in condition_id:
        return "temp_07"
    return condition_id

def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["prompt_generation"] = out["prompt_id"].map(prompt_generation) if "prompt_id" in out.columns else "unknown"
    out["prompt_base"] = out["prompt_id"].map(prompt_base) if "prompt_id" in out.columns else "unknown"
    out["condition_family_short"] = out["condition_id"].map(condition_family_from_id) if "condition_id" in out.columns else "unknown"
    out["is_control"] = out["sentence_type"].eq("control_weak_schema") if "sentence_type" in out.columns else False
    out["is_non_control"] = ~out["is_control"]

    if "schema_present" not in out.columns:
        out["schema_present"] = np.where(out.get("main_image_schema", pd.Series()).eq("NONE"), "no", "yes")
    out["gold_schema_present"] = np.where(out["is_control"], "no", "yes")

    out["schema_present_correct"] = out["schema_present"].eq(out["gold_schema_present"])
    out["primary_schema_correct"] = out["main_image_schema"].eq(out["expected_schema_primary"])
    out["lm_correct"] = out["literal_or_metaphorical"].eq(out["expected_literal_or_metaphorical"])
    out["control_correct"] = out["is_control"] & out["literal_or_metaphorical"].eq("control") & out["main_image_schema"].eq("NONE")
    out["control_false_positive_schema"] = out["is_control"] & out["main_image_schema"].notna() & ~out["main_image_schema"].eq("NONE")
    out["predicted_none"] = out["main_image_schema"].eq("NONE") | out["literal_or_metaphorical"].eq("control") | out["schema_present"].eq("no")
    return out

def pct(x, digits=1):
    if x is None or pd.isna(x):
        return "NA"
    return f"{100*x:.{digits}f}%"

def save_csv(df: pd.DataFrame, filename: str) -> Path:
    path = EXPORT_DIR / filename
    df.to_csv(path, index=False)
    print(f"Wrote: {path}")
    return path

def display_percent_table(df: pd.DataFrame, percent_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in percent_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: pct(x) if x is not None else "NA")
    return out

parsed_all = add_derived_columns(read_jsonl(PARSED_PATH))
structured = parsed_all[parsed_all["parse_status"].eq("parsed")].copy()

print(f"All parsed records: {len(parsed_all)}")
print(f"Structured records: {len(structured)}")

All parsed records: 9000
Structured records: 7200


In [2]:
# Keep only v1/v2 prompts where a paired comparison is meaningful.
pairable = structured[structured["prompt_base"].isin(["direct_schema", "structured_roles"])].copy()

key_cols = [
    "provider", "model_id", "prompt_base", "condition_id", 
    "sentence_id", "sentence_type", "expected_schema_primary",
    "expected_literal_or_metaphorical", "repetition_index"
]

v1 = pairable[pairable["prompt_generation"].eq("v1")].copy()
v2 = pairable[pairable["prompt_generation"].eq("v2_abstention")].copy()

keep_cols = key_cols + [
    "prompt_id", "schema_present", "literal_or_metaphorical", "main_image_schema",
    "schema_present_correct", "primary_schema_correct", "lm_correct",
    "control_correct", "control_false_positive_schema"
]

paired = v1[keep_cols].merge(
    v2[keep_cols],
    on=key_cols,
    how="inner",
    suffixes=("_v1", "_v2"),
)

print(f"v1 records: {len(v1)}")
print(f"v2 records: {len(v2)}")
print(f"paired records: {len(paired)}")
display(paired.head())

v1 records: 3600
v2 records: 3600
paired records: 3600


,provider,model_id,prompt_base,condition_id,sentence_id,sentence_type,expected_schema_primary,expected_literal_or_metaphorical,repetition_index,prompt_id_v1,...,control_false_positive_schema_v1,prompt_id_v2,schema_present_v2,literal_or_metaphorical_v2,main_image_schema_v2,schema_present_correct_v2,primary_schema_correct_v2,lm_correct_v2,control_correct_v2,control_false_positive_schema_v2
0,openai,openai_gpt_5_4_mini,direct_schema,c_temp_0_v1,s0001,literal_spatial,CONTAINER,literal,0,p_direct_schema_v1,...,False,p_direct_schema_v2_abstention,yes,literal,CONTAINER,True,True,True,False,False
1,openai,openai_gpt_5_4_mini,direct_schema,c_temp_0_v1,s0002,literal_spatial,CONTAINER,literal,0,p_direct_schema_v1,...,False,p_direct_schema_v2_abstention,yes,literal,SOURCE_PATH_GOAL,True,False,True,False,False
2,openai,openai_gpt_5_4_mini,direct_schema,c_temp_0_v1,s0003,literal_spatial,CONTAINER,literal,0,p_direct_schema_v1,...,False,p_direct_schema_v2_abstention,yes,literal,CONTAINER,True,True,True,False,False
3,openai,openai_gpt_5_4_mini,direct_schema,c_temp_0_v1,s0004,literal_spatial,CONTAINER,literal,0,p_direct_schema_v1,...,False,p_direct_schema_v2_abstention,yes,literal,CONTAINER,True,True,True,False,False
4,openai,openai_gpt_5_4_mini,direct_schema,c_temp_0_v1,s0005,literal_spatial,CONTAINER,literal,0,p_direct_schema_v1,...,False,p_direct_schema_v2_abstention,yes,literal,CONTAINER,True,True,True,False,False


In [3]:
# Overall paired deltas.
paired_summary = []

for group_cols in [[], ["prompt_base"], ["provider"], ["provider", "prompt_base"], ["sentence_type"], ["prompt_base", "sentence_type"]]:
    if group_cols:
        grouped = paired.groupby(group_cols, dropna=False)
    else:
        grouped = [("ALL", paired)]
    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: key for col, key in zip(group_cols, keys)}
        row["n"] = len(g)
        for metric in ["schema_present_correct", "primary_schema_correct", "lm_correct", "control_correct", "control_false_positive_schema"]:
            row[f"{metric}_v1"] = g[f"{metric}_v1"].mean()
            row[f"{metric}_v2"] = g[f"{metric}_v2"].mean()
            row[f"{metric}_delta_v2_minus_v1"] = row[f"{metric}_v2"] - row[f"{metric}_v1"]
        paired_summary.append(row)

paired_summary = pd.DataFrame(paired_summary)
save_csv(paired_summary, "paired_v1_v2_summary.csv")
display(display_percent_table(paired_summary, [c for c in paired_summary.columns if c.endswith("_v1") or c.endswith("_v2") or c.endswith("_delta_v2_minus_v1")]))

Wrote: /Users/Shared/image_schema_llm_project/data/outputs/top4_investigations/paired_v1_v2_summary.csv


,n,schema_present_correct_v1,schema_present_correct_v2,schema_present_correct_delta_v2_minus_v1,primary_schema_correct_v1,primary_schema_correct_v2,primary_schema_correct_delta_v2_minus_v1,lm_correct_v1,lm_correct_v2,lm_correct_delta_v2_minus_v1,control_correct_v1,control_correct_v2,control_correct_delta_v2_minus_v1,control_false_positive_schema_v1,control_false_positive_schema_v2,control_false_positive_schema_delta_v2_minus_v1,prompt_base,provider,sentence_type
0,3600,85.8%,92.8%,7.0%,76.1%,85.1%,9.0%,85.6%,92.6%,6.9%,18.9%,25.9%,6.9%,14.1%,7.1%,-6.9%,NaN,NaN,NaN
1,1800,80.8%,92.7%,11.9%,71.9%,85.9%,14.0%,80.8%,92.7%,11.9%,13.8%,25.9%,12.1%,19.2%,7.1%,-12.1%,direct_schema,NaN,NaN
2,1800,90.7%,92.8%,2.1%,80.3%,84.3%,3.9%,90.5%,92.4%,1.9%,24.0%,25.8%,1.8%,9.0%,7.2%,-1.8%,structured_roles,NaN,NaN
3,1200,79.8%,91.5%,11.7%,70.8%,83.6%,12.7%,79.8%,91.2%,11.4%,12.8%,24.5%,11.7%,20.2%,8.5%,-11.7%,NaN,anthropic,NaN
4,1200,79.8%,87.8%,8.1%,71.7%,80.7%,9.0%,79.8%,87.8%,8.1%,12.8%,20.8%,8.1%,20.2%,12.2%,-8.1%,NaN,google,NaN
5,1200,97.7%,98.9%,1.2%,85.9%,91.1%,5.2%,97.3%,98.7%,1.3%,31.2%,32.2%,1.1%,1.8%,0.8%,-1.1%,NaN,openai,NaN
6,600,68.5%,92.2%,23.7%,61.0%,85.8%,24.8%,68.5%,92.2%,23.7%,1.5%,25.2%,23.7%,31.5%,7.8%,-23.7%,direct_schema,anthropic,NaN
7,600,91.2%,90.8%,-0.3%,80.7%,81.3%,0.7%,91.2%,90.3%,-0.8%,24.2%,23.8%,-0.3%,8.8%,9.2%,0.3%,structured_roles,anthropic,NaN
8,600,76.8%,87.2%,10.3%,70.5%,81.2%,10.7%,76.8%,87.2%,10.3%,9.8%,20.2%,10.3%,23.2%,12.8%,-10.3%,direct_schema,google,NaN
9,600,82.7%,88.5%,5.8%,72.8%,80.2%,7.3%,82.7%,88.5%,5.8%,15.7%,21.5%,5.8%,17.3%,11.5%,-5.8%,structured_roles,google,NaN


In [4]:
# Fix/worsen transition tables for key metrics.
def transition_table(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    tab = pd.crosstab(df[f"{metric}_v1"], df[f"{metric}_v2"], rownames=[f"{metric}_v1"], colnames=[f"{metric}_v2"])
    return tab

for metric in ["control_correct", "control_false_positive_schema", "primary_schema_correct", "lm_correct", "schema_present_correct"]:
    print(f"\nTransition table: {metric}")
    display(transition_table(paired, metric))


Transition table: control_correct


control_correct_v2,False,True
control_correct_v1,,
False,2640,279
True,29,652



Transition table: control_false_positive_schema


control_false_positive_schema_v2,False,True
control_false_positive_schema_v1,,
False,3064,29
True,279,228



Transition table: primary_schema_correct


primary_schema_correct_v2,False,True
primary_schema_correct_v1,,
False,453,406
True,83,2658



Transition table: lm_correct


lm_correct_v2,False,True
lm_correct_v1,,
False,231,286
True,36,3047



Transition table: schema_present_correct


schema_present_correct_v2,False,True
schema_present_correct_v1,,
False,228,285
True,33,3054


In [5]:
# Classify paired changes into interpretable categories.
paired_changes = paired.copy()

paired_changes["control_change_type"] = np.select(
    [
        paired_changes["is_control"] if "is_control" in paired_changes.columns else paired_changes["sentence_type"].eq("control_weak_schema"),
        paired_changes["sentence_type"].eq("control_weak_schema") & paired_changes["control_false_positive_schema_v1"] & ~paired_changes["control_false_positive_schema_v2"],
        paired_changes["sentence_type"].eq("control_weak_schema") & ~paired_changes["control_false_positive_schema_v1"] & paired_changes["control_false_positive_schema_v2"],
        paired_changes["sentence_type"].eq("control_weak_schema") & paired_changes["control_correct_v1"] & paired_changes["control_correct_v2"],
        paired_changes["sentence_type"].eq("control_weak_schema") & paired_changes["control_false_positive_schema_v1"] & paired_changes["control_false_positive_schema_v2"],
    ],
    [
        "control_other",
        "v2_fixed_control_false_positive",
        "v2_introduced_control_false_positive",
        "control_correct_both",
        "control_false_positive_both",
    ],
    default="non_control",
)

# The first condition in np.select above is too broad; recode with ordered explicit logic.
def classify_control_change(row):
    if row["sentence_type"] != "control_weak_schema":
        return "non_control"
    if row["control_false_positive_schema_v1"] and not row["control_false_positive_schema_v2"]:
        return "v2_fixed_control_false_positive"
    if not row["control_false_positive_schema_v1"] and row["control_false_positive_schema_v2"]:
        return "v2_introduced_control_false_positive"
    if row["control_correct_v1"] and row["control_correct_v2"]:
        return "control_correct_both"
    if row["control_false_positive_schema_v1"] and row["control_false_positive_schema_v2"]:
        return "control_false_positive_both"
    return "control_other"

paired_changes["control_change_type"] = paired_changes.apply(classify_control_change, axis=1)

change_counts = (
    paired_changes.groupby(["prompt_base", "provider", "control_change_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["prompt_base", "provider", "control_change_type"])
)

save_csv(change_counts, "paired_v1_v2_control_change_counts.csv")
display(change_counts)

Wrote: /Users/Shared/image_schema_llm_project/data/outputs/top4_investigations/paired_v1_v2_control_change_counts.csv


,prompt_base,provider,control_change_type,count
0,direct_schema,anthropic,control_correct_both,9
1,direct_schema,anthropic,control_false_positive_both,47
2,direct_schema,anthropic,non_control,402
3,direct_schema,anthropic,v2_fixed_control_false_positive,142
4,direct_schema,google,control_correct_both,58
5,direct_schema,google,control_false_positive_both,76
6,direct_schema,google,non_control,402
7,direct_schema,google,v2_fixed_control_false_positive,63
8,direct_schema,google,v2_introduced_control_false_positive,1
9,direct_schema,openai,control_correct_both,181


In [6]:
# Export the most useful sentence-level lists for thesis examples.
fixed_controls = paired_changes[paired_changes["control_change_type"].eq("v2_fixed_control_false_positive")].copy()
introduced_controls = paired_changes[paired_changes["control_change_type"].eq("v2_introduced_control_false_positive")].copy()
still_wrong_controls = paired_changes[paired_changes["control_change_type"].eq("control_false_positive_both")].copy()

save_csv(fixed_controls, "paired_examples_v2_fixed_control_false_positives.csv")
save_csv(introduced_controls, "paired_examples_v2_introduced_control_false_positives.csv")
save_csv(still_wrong_controls, "paired_examples_control_false_positive_both.csv")

print("Examples fixed by v2:")
display(fixed_controls[[
    "provider", "model_id", "prompt_base", "sentence_id", "expected_schema_primary",
    "main_image_schema_v1", "main_image_schema_v2",
    "literal_or_metaphorical_v1", "literal_or_metaphorical_v2"
]].head(20))

Wrote: /Users/Shared/image_schema_llm_project/data/outputs/top4_investigations/paired_examples_v2_fixed_control_false_positives.csv
Wrote: /Users/Shared/image_schema_llm_project/data/outputs/top4_investigations/paired_examples_v2_introduced_control_false_positives.csv
Wrote: /Users/Shared/image_schema_llm_project/data/outputs/top4_investigations/paired_examples_control_false_positive_both.csv
Examples fixed by v2:


,provider,model_id,prompt_base,sentence_id,expected_schema_primary,main_image_schema_v1,main_image_schema_v2,literal_or_metaphorical_v1,literal_or_metaphorical_v2
128,openai,openai_gpt_5_4_mini,direct_schema,s0129,NONE,CONTAINER,NONE,literal,control
160,openai,openai_gpt_5_4_mini,direct_schema,s0161,NONE,FORCE,NONE,literal,control
163,openai,openai_gpt_5_4_mini,direct_schema,s0164,NONE,SOURCE_PATH_GOAL,NONE,literal,control
166,openai,openai_gpt_5_4_mini,direct_schema,s0167,NONE,CONTAINER,NONE,literal,control
199,openai,openai_gpt_5_4_mini,direct_schema,s0200,NONE,CONTAINER,NONE,literal,control
456,openai,openai_gpt_5_4_mini,direct_schema,s0129,NONE,CONTAINER,NONE,literal,control
457,openai,openai_gpt_5_4_mini,direct_schema,s0129,NONE,CONTAINER,NONE,literal,control
458,openai,openai_gpt_5_4_mini,direct_schema,s0130,NONE,SOURCE_PATH_GOAL,NONE,literal,control
520,openai,openai_gpt_5_4_mini,direct_schema,s0161,NONE,FORCE,NONE,literal,control
521,openai,openai_gpt_5_4_mini,direct_schema,s0161,NONE,FORCE,NONE,literal,control


In [7]:
# Visualise v1-v2 control false-positive reductions.
plot_df = paired_summary[
    paired_summary.get("prompt_base", pd.Series(index=paired_summary.index)).notna()
    & paired_summary.get("provider", pd.Series(index=paired_summary.index)).notna()
    & paired_summary.get("sentence_type", pd.Series(index=paired_summary.index)).eq("control_weak_schema")
].copy()

if not plot_df.empty:
    plot_df["label"] = plot_df["provider"] + " / " + plot_df["prompt_base"]
    ax = plot_df.plot(kind="bar", x="label", y="control_false_positive_schema_delta_v2_minus_v1", legend=False)
    ax.axhline(0)
    ax.set_title("v2 change in control false-positive schema rate")
    ax.set_xlabel("Provider / prompt family")
    ax.set_ylabel("Delta v2 - v1")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## Thesis interpretation prompts

Use this notebook to make a precise claim such as:

> The v2 abstention prompts reduced false-positive schema assignment on the exact same control items, demonstrating that the improvement is not caused by a different dataset split but by a changed prompt intervention.

Important follow-up:
- Quote 3–5 examples where v1 assigned a schema and v2 correctly returned `NONE`.
- Quote 2–3 examples where v2 still over-attributed a schema; these are theoretically valuable residual errors.